In [ ]:
from pathlib import Path
import sys

# Add parent directory to Python path
parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir))

In [ ]:
import geopandas as gpd 
import pandas as pd 

import h3_utils 
import population
import raster_utils

In [ ]:
landkreise = gpd.read_file("../aoi/Landkreise.gpkg")
gemeinden = gpd.read_file("../aoi/Gemeinden.gpkg")
metroregionen = gpd.read_file("../aoi/Metropolregionen.geojson")
stadtteile = gpd.read_file("../aoi/Gemarkungen.gpkg")

In [ ]:
f_metro = metroregionen[metroregionen["KMR"].str.contains("Frankfurt", na=False)].reset_index(drop=True)
f_metro = f_metro.to_crs(f_metro.estimate_utm_crs())

stadtteile = stadtteile.to_crs(f_metro.crs)
stadtteile["geometry"] = stadtteile["geometry"].make_valid()
stadtteile["geometry"] = stadtteile["geometry"].buffer(0)

gemeinden = gemeinden.to_crs(f_metro.crs)
gemeinden["geometry"] = gemeinden["geometry"].make_valid()
gemeinden["geometry"] = gemeinden["geometry"].buffer(0)

landkreise = landkreise.to_crs(f_metro.crs)
landkreise["geometry"] = landkreise["geometry"].make_valid()
landkreise["geometry"] = landkreise["geometry"].buffer(0)

stadtteile = stadtteile[stadtteile.centroid.intersects(f_metro.union_all())].reset_index(drop=True)
gemeinden = gemeinden[gemeinden.centroid.intersects(f_metro.union_all())].reset_index(drop=True)
landkreise = landkreise[landkreise.centroid.intersects(f_metro.union_all())].reset_index(drop=True)

In [ ]:
import geopandas as gpd
import folium

# -------------------------------------------------------
# CRS SAFE COPIES (DO NOT MODIFY ORIGINAL DATA)
# -------------------------------------------------------
f_metro_f = f_metro.to_crs("EPSG:4326")
landkreise_f = landkreise.to_crs("EPSG:4326")
gemeinden_f = gemeinden.to_crs("EPSG:4326")
stadtteile_f = stadtteile.to_crs("EPSG:4326")

# -------------------------------------------------------
# BASE MAP (BLACK & WHITE)
# -------------------------------------------------------
m = f_metro_f.explore(
    name="F-Metro",
    tiles="CartoDB positron",   # grayscale background
    control_scale=True,
    style_kwds={
        "color": "#000000",
        "weight": 10,
        "fillOpacity": 0.04,
    }
)

# -------------------------------------------------------
# LANDKREISE (medium bold, green tone)
# -------------------------------------------------------
landkreise_f.explore(
    m=m,
    name="Landkreise",
    style_kwds={
        "color": "#1b7f3a",
        "weight": 5,
        "dashArray": "12,6",
        "fillOpacity": 0.03,
    }
)

# -------------------------------------------------------
# GEMEINDEN (orange, thinner dashed)
# -------------------------------------------------------
gemeinden_f.explore(
    m=m,
    name="Gemeinden",
    style_kwds={
        "color": "#ff8c00",
        "weight": 3,
        "dashArray": "6,6",
        "fillOpacity": 0.02,
    }
)

# -------------------------------------------------------
# STADTTEILE (light red, fine pattern)
# -------------------------------------------------------
stadtteile_f.explore(
    m=m,
    name="Stadtteile",
    style_kwds={
        "color": "#cc3d3d",
        "weight": 2,
        "dashArray": "2,6",
        "fillOpacity": 0.015,
    }
)

# -------------------------------------------------------
# LAYER CONTROL
# -------------------------------------------------------
folium.LayerControl(collapsed=False).add_to(m)

# -------------------------------------------------------
# LEGEND (CUSTOM HTML)
# -------------------------------------------------------
legend_html = """
<div style="
position: fixed;
bottom: 40px;
left: 40px;
z-index: 9999;
background: white;
padding: 12px;
border: 2px solid grey;
font-size: 13px;
border-radius: 6px;
">

<b>Administrative Hierarchy</b><br><br>

<span style="color:#000; font-weight:bold;">━━━━</span> F-Metro<br>
<span style="color:#1b7f3a;">- - - - -</span> Landkreise<br>
<span style="color:#ff8c00;">- - - - -</span> Gemeinden<br>
<span style="color:#cc3d3d;">········</span> Stadtteile<br>

</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))
m.save("region_map.html")
m

In [ ]:
city = landkreise[landkreise["KREIS_BZ"].str.contains("Frankfurt", na=False)].reset_index(drop=True)
city

In [ ]:
suburban_area = landkreise[landkreise.centroid.intersects(city.union_all().buffer(10000))].reset_index(drop=True)
suburban_area

In [ ]:
# ----------------------------
# 1. AOI parts from existing layers
# ----------------------------

stadtteile_part = stadtteile.loc[
    stadtteile.centroid.intersects(city.union_all()),
    ["geometry", "GMK_BZ"]
].rename(columns={"GMK_BZ": "Name"})
stadtteile_part["type"] = "Stadtteil"

gemeinden_part = gemeinden.loc[
    gemeinden.centroid.intersects(
        suburban_area.union_all().difference(city.union_all())
    ),
    ["geometry", "GMDE_BZ"]
].rename(columns={"GMDE_BZ": "Name"})
gemeinden_part["type"] = "Gemeinde"

landkreise_part = landkreise.loc[
    ~landkreise.centroid.intersects(suburban_area.union_all()),
    ["geometry", "KREIS_BZ"]
].rename(columns={"KREIS_BZ": "Name"})
landkreise_part["type"] = "Landkreis"

# Combine AOI
aoi = pd.concat(
    [stadtteile_part, gemeinden_part, landkreise_part],
    ignore_index=True
)

# ----------------------------
# 2. Split MultiPolygon (Rheinland-Pfalz / Bayern)
# ----------------------------

geom_raw = f_metro.difference(landkreise.union_all()).union_all()

parts = [g for g in geom_raw.geoms if g.area > 10**9]

metro_parts = gpd.GeoDataFrame(
    {"geometry": parts},
    crs=stadtteile.crs
)

metro_parts["Name"] = ["Rheinland-Pfalz", "Bayern"]
metro_parts["type"] = "Bundesland"

# ----------------------------
# 3. Final merge
# ----------------------------

aoi = pd.concat([aoi, metro_parts], ignore_index=True)

aoi = gpd.GeoDataFrame(aoi, geometry="geometry", crs=stadtteile.crs)
aoi = aoi[aoi["Name"] != "Main"].reset_index(drop=True)

In [ ]:
aoi.to_file("../aoi/aoi.gpkg")

In [ ]:
population_file = population.download_worldpop_population(
    aoi,
    2025,
    folder="../population",
    resolution="100m",
)

In [ ]:
for i in range(len(aoi)):
    aoi_i = aoi.iloc[i:i+1] 
    pop, transform, crs = raster_utils.read_raster(population_file,aoi=aoi_i.to_crs(4326))
    pop_gdf = raster_utils.vectorize(pop, transform, crs)
    pop_gdf = pop_gdf.to_crs(aoi.crs)
    pop_gdf = pop_gdf[pop_gdf.centroid.intersects(aoi_i.union_all())].reset_index(drop=True)
    aoi.loc[i,"population"] = pop_gdf["value"].sum()

In [ ]:
aoi.to_file("../aoi/aoi.gpkg")

In [ ]:
aoi